In [1]:
#%pip install -U "torch>=2.6" "transformers>=4.50,<5.0" datasets scikit-learn seqeval pandas numpy Faker
#%pip install -U ipywidgets jupyterlab
#%pip install hf_xet
# %pip install "accelerate>=0.26.0"  # один раз при настройке окружения
#%pip install -q pymorphy3
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())

Python: C:\Users\Vlad\AppData\Local\Programs\Python\Python312\python.exe
PyTorch: 2.14.1+cpu
CUDA: None
GPU: False


In [2]:
import gc
import os
import re
import torch
import random
import zipfile
import pymorphy3
import numpy as np
import pandas as pd
import urllib.request
from faker import Faker
from pathlib import Path
from seqeval.scheme import IOB2
from datetime import date, datetime
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, concatenate_datasets
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForTokenClassification, set_seed, pipeline
from seqeval.metrics import precision_score as ner_precision, recall_score as ner_recall, f1_score as ner_f1

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Устройство: {device}")

PyTorch: 2.14.1+cpu
Устройство: cpu


In [3]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"
ENTITY_TYPES = [
    "NAME","SURNAME","PATRONYMIC",
    "EMAIL",
    "INN","PHONE","CARD","SNILS",
    "BIRTH_DATE"
]

# BIO
LABELS = ["O"]
for entity in ENTITY_TYPES:
    LABELS.extend([f"B-{entity}", f"I-{entity}"])
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

#Токенайзер, модель
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print(f"Модель: {MODEL_NAME}")
print(f"Количество атрибутов: {len(ENTITY_TYPES)}")
print(f"Количество BIO-меток: {len(LABELS)}")
print(f"Устройство: {device}")
print(f"Fast tokenizer: {tokenizer.is_fast}")

Модель: DeepPavlov/rubert-base-cased
Количество атрибутов: 9
Количество BIO-меток: 19
Устройство: cpu
Fast tokenizer: True


In [4]:
#Загрузка датасета factRuEval-2016
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = DATA_DIR / "factRuEval-2016.zip"
DATASET_PATH = DATA_DIR / "factRuEval-2016-master"

DATASET_URL = (
    "https://github.com/dialogue-evaluation/"
    "factRuEval-2016/archive/master.zip"
)

if not DATASET_PATH.exists():
    if not ARCHIVE_PATH.exists():
        print("Скачивание factRuEval-2016...")
        urllib.request.urlretrieve(
            DATASET_URL,
            ARCHIVE_PATH
        )

    print("Распаковка архива...")
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

print("Датасет доступен:", DATASET_PATH)
print("Devset:", (DATASET_PATH / "devset").exists())
print("Testset:", (DATASET_PATH / "testset").exists())

Датасет доступен: C:\Users\Vlad\1JupyterLab\CyberSec\data\factRuEval-2016-master
Devset: True
Testset: True


In [5]:
#Чтение
def load_factrueval(folder):
    records = []
    for txt_path in sorted(folder.glob("*.txt")):
        text = txt_path.read_text(encoding="utf-8-sig")
        spans_path = txt_path.with_suffix(".spans")
        tokens_path = txt_path.with_suffix(".tokens")
        spans = []
        tokens = []
        #Фрагменты
        if spans_path.exists():
            for line in spans_path.read_text(encoding="utf-8-sig").splitlines():
                fields = line.split("#", 1)[0].split()
                if len(fields) < 6:
                    continue
                span_id, label, start, length = fields[:4]
                spans.append({"label": label, "start": int(start), "end": int(start) + int(length)})
        #Токены        
        if tokens_path.exists():
            for line in tokens_path.read_text(encoding="utf-8-sig").splitlines():
                fields = line.split(maxsplit=3)
                if len(fields) < 4:
                    continue
                token_id, start, length, word = fields
                tokens.append({"text": word,"start": int(start),"end": int(start) + int(length)})
                
        records.append({
            "id": txt_path.stem,
            "text": text,
            "tokens": tokens,
            "spans": spans
        })
    return records

dev_records = load_factrueval(DATASET_PATH / "devset")
test_records = load_factrueval(DATASET_PATH / "testset")

# Разделение devset на обучение и валидацию
train_records, val_records = train_test_split(
    dev_records,
    test_size=0.2,
    random_state=SEED
)
datasets = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
    "test": Dataset.from_list(test_records)
})

print(datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'spans'],
        num_rows: 97
    })
    validation: Dataset({
        features: ['id', 'text', 'tokens', 'spans'],
        num_rows: 25
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'spans'],
        num_rows: 133
    })
})


In [6]:

print("Обучающих документов:", len(datasets["train"]))
print("Валидационных документов:", len(datasets["validation"]))
print("Тестовых документов:", len(datasets["test"]))

sample = datasets["train"][0]

print("\nID документа:", sample["id"])
print("\nФрагмент текста:")
print(sample["text"][:500])

print("\nПервые 10 размеченных сущностей:")

for span in sample["spans"][:10]:
    start = span["start"]
    end = span["end"]

    print({
        "label": span["label"],
        "text": sample["text"][start:end],
        "start": start,
        "end": end
    })


Обучающих документов: 97
Валидационных документов: 25
Тестовых документов: 133

ID документа: book_288

Фрагмент текста:
5 апреля 2011 года столице Чеченской республики городе Грозном прошла церемония инаугурации экс-боевика сепаратистов, а ныне генерал-майора МВД Рамзана Кадырова. Ещё 28 февраля президент Российской Федерации Дмитрий Анатольевич Медведев предложил парламенту Чечни рассмотреть кандидатуру действующего ставленника Кремля, с целью избрания его на второй срок. Как это не удивительно, чеченские парламентарии (в подовляющем большинстве близкие клану Кадырова люди) восприняли инициативу Москвы с энтузиа

Первые 10 размеченных сущностей:
{'label': 'loc_descr', 'text': 'республики', 'start': 37, 'end': 47}
{'label': 'loc_name', 'text': 'Чеченской республики', 'start': 27, 'end': 47}
{'label': 'loc_descr', 'text': 'городе', 'start': 48, 'end': 54}
{'label': 'loc_name', 'text': 'Грозном', 'start': 55, 'end': 62}
{'label': 'org_name', 'text': 'МВД', 'start': 140, 'end': 143}
{'la

In [7]:
#Регулярки
STRUCTURED_TYPES = ("EMAIL","PHONE","INN","CARD","SNILS","BIRTH_DATE")
PATTERNS = {
    "EMAIL": re.compile( # [!a1_.@] [буквы+спецсимволы] [@] [буквы,цифры,дефисы] [.] [минимум две буквы-цифры] [!a1_+]
        r"(?<![\w.@])[\w.%+\-]+@(?:[A-Za-z0-9\-]+\.)+[A-Za-z]{2,}(?![\w@-])", re.I),
    "PHONE": re.compile(
        r"(?<![\w+])(?:\+7|8)[ \t-]?\(?[3489]\d{2}\)?"   # [!a1_+] [+7|8] [ |/t|-] [(] [3|4|8|9] [xx] [)]
        r"[ \t-]?\d{3}[ \t-]?\d{2}[ \t-]?\d{2}(?![\w-])" # [ |/t|-] [xxx] [ |/t|-] [xx] [ |/t|-] [xx] [!a1_+]
    ),                                                   # неподдерж вне-рф, и "79231116145" 
    "INN": re.compile(r"(?<!\d)\d{12}(?!\d)"),  #[!1][xxxxxxxxxxxx][!1]
    "CARD": re.compile(r"(?<!\d)\d(?:[ -]?\d){12,18}(?![\d-])"),  #visa 13,16,19, Mastercard 16, AMEX 15, MIR 16-19
    "SNILS": re.compile(r"(?<!\d)(?:\d{3}[- ]\d{3}[- ]\d{3}[- ]\d{2}|\d{11})(?!\d)"),  #[!1] [xxx] [ |-] [xxx] [ |-] [xxx] [ |-] [xx] [!1]
    "BIRTH_DATE": re.compile(r"(?<!\d)(?:\d{2}[./-]\d{2}[./-]\d{4}|\d{4}-\d{2}-\d{2})(?!\d)") #[!1] [xx] [./-] [xx] [./-] [xxxx]
}


In [8]:
#Контрольные суммы
#Удаляет все символы, кроме цифр.
def digits_only(value: str) -> str:
    return re.sub(r"\D", "", value)

#Проверка контрольной суммы алгоритмом Луна.
#1. Цифры проверяемой последовательности нумеруются справа налево.
#2. Цифры, оказавшиеся на нечётных местах, остаются без изменений.
#3. Цифры, стоящие на чётных местах, умножаются на 2.
#4. Если в результате такого умножения возникает число больше 9, 
    #оно заменяется суммой цифр получившегося произведения — однозначным числом, то есть цифрой.
#5. Все полученные в результате преобразования цифры складываются. Если сумма кратна 10, то исходные данные верны.
def valid_luhn(value: str) -> bool:
    number = digits_only(value)
    if (not number.isascii() or not number.isdigit() or not 13 <= len(number) <= 19):
        return False
    total = 0
    
    for index, char in enumerate(reversed(number)):
        digit = int(char)
        if index % 2:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0

#Проверка номера банковской карты.
def valid_card(value: str) -> bool:
    number = digits_only(value)

    return (
        13 <= len(number) <= 19  # ну вдруг
        and number[0] in "23456" # 2 мир, мастеркард, 3 amex, 4 visa, 5 mastercard, 6 discover, unionpay
        and len(set(number)) > 1
        and valid_luhn(number)
    )

#Проверка 12-значного ИНН физического лица.
def valid_inn(value: str) -> bool:
    if not re.fullmatch(r"[0-9]{12}", value):
        return False
    numbers = list(map(int, value))
    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)    #веса для расчёта контрольной суммы
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    c11 = sum(n * w for n, w in zip(numbers[:10], w11)) % 11 % 10
    c12 = sum(n * w for n, w in zip(numbers[:11], w12)) % 11 % 10
    return (c11, c12) == (numbers[10], numbers[11])

#Проверка контрольного числа СНИЛС.
def valid_snils(value: str) -> bool:
    number = digits_only(value)
    if (len(number) != 11 or not number.isascii() or not number.isdigit()):
        return False
    if int(number[:9]) <= 1001998:   # снилс менее 001-001-998 считается по другому, отметаем
        return False

    weighted_sum = sum(int(c) * (9 - i) for i, c in enumerate(number[:9]))
    control = weighted_sum % 101
    if control == 100:
        control = 0
    return control == int(number[-2:])

#Проверяет корректность календарной даты.
def valid_birth_date(value: str) -> bool:
    formats = ("%d.%m.%Y","%d/%m/%Y","%d-%m-%Y","%Y-%m-%d")
    for fmt in formats:
        try:
            parsed = datetime.strptime(value, fmt).date()
            return (
                date(1900, 1, 1)
                <= parsed
                <= date.today()
            )
        except ValueError:
            continue
    return False

In [9]:
#Определяем контекст даты рождения.
def birth_context(text: str, start: int, end: int) -> bool:
    before = text[max(0, start - 60):start]
    after = text[end:min(len(text), end + 20)]

    lead = re.search(
        r"(?:дата\s+рождения|д\.\s*р\.?|"
        r"родил[а-яё]*|день\s+рождения)"
        r"\s*[:\-]?\s*$",
        before,
        re.I
    )

    trail = re.match(
        r"\s*(?:г\.?\s*р\.?|года\s+рождения)"
        r"(?![а-яё\w])",
        after,
        re.I
    )

    return bool(lead or trail)


In [10]:
# Структурированные ПДн
def detect_structured(text: str) -> list[list]:
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    validators = {
        "INN": valid_inn,
        "CARD": valid_card,
        "SNILS": valid_snils,
        "BIRTH_DATE": valid_birth_date
    }

    priority = {
        "EMAIL": 6,
        "PHONE": 5,
        "CARD": 4,
        "INN": 3,
        "SNILS": 2,
        "BIRTH_DATE": 1
    }

    candidates = []
    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            start, end = match.span()
            value = match.group()

            # Проверка контрольных разрядов и даты
            if label in validators:
                if not validators[label](value):
                    continue

            # Для СНИЛС без разделителей
            # требуем явное указание типа
            if label == "SNILS" and value.isdigit():
                preceding = text[max(0, start - 25):start]

                if not re.search(
                    r"снилс\s*[:№-]?\s*$",
                    preceding,
                    re.I
                ):
                    continue

            # Дата должна быть датой рождения
            if label == "BIRTH_DATE":
                if not birth_context(text, start, end):
                    continue

            # Проверка длины российского телефона
            if label == "PHONE":
                if len(digits_only(value)) != 11:
                    continue

            candidates.append((label, start, end))

    # Сортировка по приоритету
    candidates.sort(
        key=lambda x: (
            -priority[x[0]],
            -(x[2] - x[1]),
            x[1]
        )
    )

    # Исключение пересекающихся сущностей
    selected = []

    for label, start, end in candidates:
        overlap = any(
            start < e and s < end
            for _, s, e in selected
        )

        if not overlap:
            selected.append((label, start, end))

    # Сортировка по позиции в тексте
    selected.sort(key=lambda x: (x[1], x[2]))

    return [
        [label, start, end]
        for label, start, end in selected
    ]


In [11]:
# проверка
text = (
    "Контакт: ivan.petrov@example.com; "
    "телефон +7 (999) 123-45-67; "
    "ИНН 770698959438; "
    "карта 4111 1111 1111 1111; "
    "СНИЛС 112-233-445 95; "
    "дата рождения 15.04.1990. "
    "Встреча 05.04.2011. "
    "Неверная карта 4111 1111 1111 1112."
)

entities = detect_structured(text)

for label, start, end in entities:
    print(
        f"{label:12} | "
        f"{text[start:end]:26} | "
        f"[{start}, {end}]"
    )

print("\nРезультат:")
print(entities)


EMAIL        | ivan.petrov@example.com    | [9, 32]
PHONE        | +7 (999) 123-45-67         | [42, 60]
INN          | 770698959438               | [66, 78]
CARD         | 4111 1111 1111 1111        | [86, 105]
SNILS        | 112-233-445 95             | [113, 127]
BIRTH_DATE   | 15.04.1990                 | [143, 153]

Результат:
[['EMAIL', 9, 32], ['PHONE', 42, 60], ['INN', 66, 78], ['CARD', 86, 105], ['SNILS', 113, 127], ['BIRTH_DATE', 143, 153]]


In [12]:

import random
from faker import Faker

rng = random.Random(42)
fake = Faker("ru_RU")
Faker.seed(42)


def make_test_inn():
    base = "7701" + "".join(
        str(rng.randrange(10)) for _ in range(6)
    )

    d = list(map(int, base))

    w1 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    w2 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)

    c1 = sum(a*b for a, b in zip(d, w1)) % 11 % 10
    c2 = sum(a*b for a, b in zip(d+[c1], w2)) % 11 % 10

    return base + str(c1) + str(c2)


def make_test_snils():
    base = str(
        rng.randrange(1_001_999, 1_000_000_000)
    ).zfill(9)

    control = sum(
        int(c) * (9-i)
        for i, c in enumerate(base)
    ) % 101

    if control == 100:
        control = 0

    return (
        f"{base[:3]}-{base[3:6]}-"
        f"{base[6:9]} {control:02d}"
    )


structured_tests = []

for label in STRUCTURED_TYPES:
    for i in range(50):

        if label == "EMAIL":
            value = fake.email()
            prefix = "Почта: "

        elif label == "PHONE":
            v = "9" + "".join(
                str(rng.randrange(10))
                for _ in range(9)
            )
            value = (
                f"+7 ({v[:3]}) {v[3:6]}-"
                f"{v[6:8]}-{v[8:10]}"
            )
            prefix = "Телефон: "

        elif label == "INN":
            value = make_test_inn()
            prefix = "ИНН: "

        elif label == "CARD":
            number = fake.credit_card_number(
                card_type="visa"
            )
            value = " ".join(
                number[j:j+4]
                for j in range(0, len(number), 4)
            )
            prefix = "Карта: "

        elif label == "SNILS":
            value = make_test_snils()
            prefix = "СНИЛС: "

        elif label == "BIRTH_DATE":
            value = fake.date_of_birth(
                minimum_age=18,
                maximum_age=80
            ).strftime("%d.%m.%Y")
            prefix = "Дата рождения: "

        text_test = f"Анкета № {i}. {prefix}{value}."
        start = text_test.index(value)

        expected = [
            [label, start, start + len(value)]
        ]

        predicted = detect_structured(text_test)

        structured_tests.append({
            "text": text_test,
            "label": label,
            "expected": expected,
            "predicted": predicted,
            "correct": expected == predicted
        })

tests_df = pd.DataFrame(structured_tests)

print("Всего текстов:", len(tests_df))
print("\nРезультаты по атрибутам:")

print(
    tests_df.groupby("label")["correct"].agg(
        ["count", "sum", "mean"]
    )
)

assert tests_df["correct"].all(), \
    "Обнаружены ошибки на синтетических примерах"


Всего текстов: 300

Результаты по атрибутам:
            count  sum  mean
label                       
BIRTH_DATE     50   50   1.0
CARD           50   50   1.0
EMAIL          50   50   1.0
INN            50   50   1.0
PHONE          50   50   1.0
SNILS          50   50   1.0


In [13]:
# BIO разметка
# Соответствие разметки factRuEval нашим атрибутам
PERSON_TYPES = {
    "name": "NAME",
    "surname": "SURNAME",
    "patronymic": "PATRONYMIC"
}


def create_bio_labels(example):
    """Преобразует spans и tokens документа в BIO-разметку."""

    tokens = example["tokens"]
    spans = example["spans"]

    # Исходные слова и их координаты
    words = [token["text"] for token in tokens]

    offsets = [
        [int(token["start"]), int(token["end"])]
        for token in tokens
    ]

    # Сначала каждый токен не является ПДн
    bio_tags = ["O"] * len(tokens)

    # Сопоставление символьных границ с индексами токенов
    start_to_index = {
        int(token["start"]): i
        for i, token in enumerate(tokens)
    }

    end_to_index = {
        int(token["end"]): i
        for i, token in enumerate(tokens)
    }

    skipped = 0
    annotated = 0

    # Сортируем сущности по расположению в тексте
    for span in sorted(
        spans,
        key=lambda s: (int(s["start"]), int(s["end"]))
    ):
        entity_type = PERSON_TYPES.get(
            span["label"].lower()
        )

        # Остальные типы не размечаем для RuBERT
        if entity_type is None:
            continue

        start = int(span["start"])
        end = int(span["end"])

        first = start_to_index.get(start)
        last = end_to_index.get(end)

        # Пропускаем несогласованные границы
        if first is None or last is None or first > last:
            skipped += 1
            continue

        indices = range(first, last + 1)

        # Один токен не может принадлежать двум сущностям
        if any(bio_tags[i] != "O" for i in indices):
            skipped += 1
            continue

        # Начало сущности
        bio_tags[first] = f"B-{entity_type}"

        # Продолжение сущности
        for i in range(first + 1, last + 1):
            bio_tags[i] = f"I-{entity_type}"

        annotated += 1

    # Переводим строковые BIO-метки в числовые
    ner_tags = [label2id[tag] for tag in bio_tags]

    return {
        "words": words,
        "word_offsets": offsets,
        "bio_tags": bio_tags,
        "ner_tags": ner_tags,
        "bio_annotated": annotated,
        "bio_skipped": skipped
    }


In [14]:
#Применяем BIO ко вему датасету

bio_datasets = datasets.map(
    create_bio_labels,
    desc="Создание BIO-разметки"
)

print(bio_datasets)

for split in ["train", "validation", "test"]:
    data = bio_datasets[split]

    total_entities = sum(data["bio_annotated"])
    skipped_entities = sum(data["bio_skipped"])

    print(f"\n{split}:")
    print("Документов:", len(data))
    print("Размеченных сущностей:", total_entities)
    print("Пропущенных сущностей:", skipped_entities)


Создание BIO-разметки:   0%|          | 0/97 [00:00<?, ? examples/s]

Создание BIO-разметки:   0%|          | 0/25 [00:00<?, ? examples/s]

Создание BIO-разметки:   0%|          | 0/133 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 97
    })
    validation: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 25
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 133
    })
})

train:
Документов: 97
Размеченных сущностей: 1043
Пропущенных сущностей: 2

validation:
Документов: 25
Размеченных сущностей: 192
Пропущенных сущностей: 0

test:
Документов: 133
Размеченных сущностей: 2105
Пропущенных сущностей: 0


In [15]:

sample = next(
    item for item in bio_datasets["train"]
    if item["id"] == "book_288"
)

# Найдём первое персональное имя
first_entity = next(
    i for i, tag in enumerate(sample["bio_tags"])
    if tag != "O"
)

left = max(0, first_entity - 5)
right = min(len(sample["words"]), first_entity + 15)

preview = pd.DataFrame({
    "Token": sample["words"][left:right],
    "BIO": sample["bio_tags"][left:right],
    "Label ID": sample["ner_tags"][left:right],
    "Start": [
        p[0] for p in sample["word_offsets"][left:right]
    ],
    "End": [
        p[1] for p in sample["word_offsets"][left:right]
    ]
})

display(preview)


,Token,BIO,Label ID,Start,End
0,",",O,0,116,117
1,а,O,0,118,119
2,ныне,O,0,120,124
3,генерал-майора,O,0,125,139
4,МВД,O,0,140,143
5,Рамзана,B-NAME,1,144,151
6,Кадырова,B-SURNAME,3,152,160
7,.,O,0,160,161
8,Ещё,O,0,162,165
9,28,O,0,166,168


In [16]:

def tokenize_and_align(words, bio_tags):
    """Токенизация WordPiece с согласованием BIO-меток."""

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        truncation=False
    )

    word_ids = encoded.word_ids()

    aligned_labels = []
    previous_word = None

    for word_id in word_ids:

        if word_id is None:
            # [CLS], [SEP] и другие специальные токены
            aligned_labels.append(-100)

        else:
            tag = bio_tags[word_id]

            # Продолжение того же слова после WordPiece
            if word_id == previous_word and tag.startswith("B-"):
                tag = "I-" + tag[2:]

            aligned_labels.append(label2id[tag])

        previous_word = word_id

    encoded["labels"] = aligned_labels

    return encoded


In [17]:

example_words = sample["words"][left:right]
example_tags = sample["bio_tags"][left:right]

encoded = tokenize_and_align(
    example_words,
    example_tags
)

subtokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"]
)

aligned_tags = [
    id2label[label] if label != -100 else "IGNORE"
    for label in encoded["labels"]
]

result = pd.DataFrame({
    "RuBERT Token": subtokens,
    "BIO": aligned_tags,
    "Label ID": encoded["labels"]
})

display(result)

assert len(encoded["input_ids"]) == len(encoded["labels"])


,RuBERT Token,BIO,Label ID
0,[CLS],IGNORE,-100
1,",",O,0
2,а,O,0
3,ныне,O,0
4,генерал,O,0
5,-,O,0
6,майора,O,0
7,МВД,O,0
8,Рамзана,B-NAME,1
9,Кадырова,B-SURNAME,3


In [18]:

test_counts = {}

for entity in ["NAME", "SURNAME", "PATRONYMIC"]:
    target_tag = f"B-{entity}"

    count = sum(
        target_tag in record["bio_tags"]
        for record in bio_datasets["test"]
    )

    test_counts[entity] = count

    print(
        f"{entity}: {count} тестовых документов "
        f"с минимум одним упоминанием"
    )

print("\nТребование 50 текстов:")

for entity, count in test_counts.items():
    print(
        entity,
        "OK" if count >= 50 else "Нужно дополнить"
    )


NAME: 126 тестовых документов с минимум одним упоминанием
SURNAME: 125 тестовых документов с минимум одним упоминанием
PATRONYMIC: 11 тестовых документов с минимум одним упоминанием

Требование 50 текстов:
NAME OK
SURNAME OK
PATRONYMIC Нужно дополнить


In [19]:
#Дополняем через фейкер
person_fake = Faker("ru_RU")
person_fake.seed_instance(2026)
person_rng = random.Random(2026)

# Разные контексты упоминания ФИО
TEMPLATES = [
    "В анкете указано: {SURNAME} {NAME} {PATRONYMIC}.",
    "Контактное лицо: {NAME} {PATRONYMIC} {SURNAME}.",
    "В документе встречается запись: {SURNAME} {NAME} {PATRONYMIC}.",
    "В списке участников: {SURNAME} {NAME} {PATRONYMIC}.",
    "Указанное ФИО: {NAME} {PATRONYMIC} {SURNAME}.",
    "В карточке клиента записано: {SURNAME} {NAME} {PATRONYMIC}.",
    "Ответственное лицо: {NAME} {PATRONYMIC} {SURNAME}.",
    "Имя заявителя в реестре: {SURNAME} {NAME} {PATRONYMIC}."
]

PLACEHOLDER_RE = re.compile(
    r"\{(SURNAME|NAME|PATRONYMIC)\}"
)

WORD_RE = re.compile(
    r"[^\W\d_]+(?:-[^\W\d_]+)*|\d+|[^\w\s]"
)


def create_synthetic_person(split, i):
    gender = person_rng.choice(["male", "female"])

    values = {
        "NAME": getattr(person_fake, f"first_name_{gender}")(),
        "SURNAME": getattr(person_fake, f"last_name_{gender}")(),
        "PATRONYMIC": getattr(person_fake, f"middle_name_{gender}")()
    }

    template = person_rng.choice(TEMPLATES)

    parts = []
    spans = []
    position = 0
    cursor = 0

    for match in PLACEHOLDER_RE.finditer(template):
        prefix = template[cursor:match.start()]

        parts.append(prefix)
        position += len(prefix)

        label = match.group(1)
        value = values[label]

        spans.append({
            "label": label.lower(),
            "start": position,
            "end": position + len(value)
        })

        parts.append(value)
        position += len(value)
        cursor = match.end()

    parts.append(template[cursor:])
    text = "".join(parts)

    tokens = [
        {
            "text": match.group(),
            "start": match.start(),
            "end": match.end()
        }
        for match in WORD_RE.finditer(text)
    ]

    return {
        "id": f"synthetic_{split}_{i:04d}",
        "text": text,
        "tokens": tokens,
        "spans": spans
    }


# Отдельная генерация выборок
synthetic_records = {
    "train": [
        create_synthetic_person("train", i)
        for i in range(300)
    ],
    "validation": [
        create_synthetic_person("validation", i)
        for i in range(60)
    ],
    "test": [
        create_synthetic_person("test", i)
        for i in range(60)
    ]
}

synthetic_datasets = DatasetDict({
    split: Dataset.from_list(records)
    for split, records in synthetic_records.items()
})

# Применяем существующую BIO-разметку
synthetic_bio = synthetic_datasets.map(
    create_bio_labels
)

for split in synthetic_bio:
    data = synthetic_bio[split]

    print(
        split,
        "| текстов:", len(data),
        "| сущностей:", sum(data["bio_annotated"]),
        "| пропущено:", sum(data["bio_skipped"])
    )

    assert sum(data["bio_skipped"]) == 0


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

train | текстов: 300 | сущностей: 900 | пропущено: 0
validation | текстов: 60 | сущностей: 180 | пропущено: 0
test | текстов: 60 | сущностей: 180 | пропущено: 0


In [20]:
#Объединяем с rueval

combined_bio = DatasetDict({
    split: concatenate_datasets([
        bio_datasets[split],
        synthetic_bio[split]
    ])
    for split in ["train", "validation", "test"]
})

print(combined_bio)

for split in combined_bio:
    print(split, len(combined_bio[split]))


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 397
    })
    validation: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 85
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'spans', 'words', 'word_offsets', 'bio_tags', 'ner_tags', 'bio_annotated', 'bio_skipped'],
        num_rows: 193
    })
})
train 397
validation 85
test 193


In [21]:

for entity in ["NAME", "SURNAME", "PATRONYMIC"]:
    count = sum(
        f"B-{entity}" in record["bio_tags"]
        for record in combined_bio["test"]
    )

    print(f"{entity}: {count} тестовых текстов")


NAME: 186 тестовых текстов
SURNAME: 185 тестовых текстов
PATRONYMIC: 71 тестовых текстов


In [22]:
#Разбивка на последовательности до 512
PERSON_LABELS = ["O","B-NAME", "I-NAME","B-SURNAME", "I-SURNAME","B-PATRONYMIC", "I-PATRONYMIC"]

person_label2id = {
    label: i for i, label in enumerate(PERSON_LABELS)
}

person_id2label = {
    i: label for i, label in enumerate(PERSON_LABELS)
}

MAX_LENGTH = 512

print("Количество классов RuBERT:", len(PERSON_LABELS))
print("Максимальная длина:", MAX_LENGTH)


Количество классов RuBERT: 7
Максимальная длина: 512


In [23]:
# Разбивает документ на окна с BIO-разметкой
def prepare_ner_windows(record):


    words = record["words"]
    tags = record["bio_tags"]

    if not words:
        return []

    assert len(words) == len(tags)

    # Сколько WordPiece-токенов занимает каждое слово
    full_encoding = tokenizer(
        words,
        is_split_into_words=True,
        add_special_tokens=False,
        truncation=False
    )

    piece_counts = [0] * len(words)

    for word_id in full_encoding.word_ids():
        if word_id is not None:
            piece_counts[word_id] += 1

    special_count = tokenizer.num_special_tokens_to_add(
        pair=False
    )

    available = MAX_LENGTH - special_count

    if any(count == 0 or count > available for count in piece_counts):
        raise ValueError(
            f"Некорректная длина токена в {record['id']}"
        )

    windows = []
    start = 0

    while start < len(words):
        end = start
        used = 0

        # Собираем максимально возможное окно
        while (
            end < len(words)
            and used + piece_counts[end] <= available
        ):
            used += piece_counts[end]
            end += 1

        # Стараемся не разрезать BIO-сущность
        if end < len(words) and tags[end].startswith("I-"):
            boundary = end - 1

            while (
                boundary >= start
                and tags[boundary].startswith("I-")
            ):
                boundary -= 1

            if (
                boundary > start
                and tags[boundary].startswith("B-")
            ):
                end = boundary

        chunk_words = words[start:end]
        chunk_tags = tags[start:end].copy()

        # Если длинную сущность пришлось разделить
        if chunk_tags[0].startswith("I-"):
            chunk_tags[0] = "B-" + chunk_tags[0][2:]

        encoded = tokenizer(
            chunk_words,
            is_split_into_words=True,
            add_special_tokens=True,
            truncation=False
        )

        # Присваиваем метку первому подслову
        # Остальным частям того же слова ставим -100
        aligned_labels = []
        previous_word = None

        for word_id in encoded.word_ids():
            if word_id is None:
                aligned_labels.append(-100)

            elif word_id != previous_word:
                tag = chunk_tags[word_id]
                aligned_labels.append(person_label2id[tag])

            else:
                aligned_labels.append(-100)

            previous_word = word_id

        assert len(encoded["input_ids"]) <= MAX_LENGTH
        assert len(encoded["input_ids"]) == len(aligned_labels)

        windows.append({
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": aligned_labels
        })

        start = end

    return windows


In [24]:

from datasets import Dataset, DatasetDict

window_datasets = {}

for split in ["train", "validation", "test"]:
    all_windows = []

    for record in combined_bio[split]:
        windows = prepare_ner_windows(record)
        all_windows.extend(windows)

    window_datasets[split] = Dataset.from_list(
        all_windows
    )

    print(
        split,
        "| документов:", len(combined_bio[split]),
        "| окон:", len(all_windows)
    )

ner_datasets = DatasetDict(window_datasets)
base_train_windows = ner_datasets["train"]
print("\nПодготовленные данные:")
print(ner_datasets)


train | документов: 397 | окон: 413
validation | документов: 85 | окон: 85
test | документов: 193 | окон: 252

Подготовленные данные:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 413
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 85
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 252
    })
})


In [25]:

for split in ner_datasets:
    lengths = [
        len(row["input_ids"])
        for row in ner_datasets[split]
    ]

    assert max(lengths) <= MAX_LENGTH

    print(
        split,
        "| максимальная длина:", max(lengths),
        "| минимальная длина:", min(lengths)
    )

print("\nВсе последовательности подготовлены корректно.")


train | максимальная длина: 512 | минимальная длина: 10
validation | максимальная длина: 437 | минимальная длина: 10
test | максимальная длина: 512 | минимальная длина: 10

Все последовательности подготовлены корректно.


In [26]:
# Проверяем сохранённые модели перед запуском обучения.
# Пути те же, что и при сохранении в исходном ноутбуке.
BASE_MODEL_DIR = Path.cwd() / "rubert_pii_fio"
EXTRA_MODEL_DIR = Path.cwd() / "rubert_pii_fio_extra"


def saved_rubert_exists(folder):
    folder = Path(folder)
    has_config = (folder / "config.json").is_file()
    has_weights = any((folder / name).is_file() for name in (
        "model.safetensors", "pytorch_model.bin",
        "model.safetensors.index.json", "pytorch_model.bin.index.json"
    ))
    return has_config and has_weights


BASE_SAVED = saved_rubert_exists(BASE_MODEL_DIR)
EXTRA_SAVED = saved_rubert_exists(EXTRA_MODEL_DIR)
TRAIN_BASE = not BASE_SAVED and not EXTRA_SAVED
TRAIN_EXTRA = not EXTRA_SAVED

print(f"Базовая модель: {'сохранена' if BASE_SAVED else 'не найдена'} ({BASE_MODEL_DIR})")
print(f"Дообученная модель: {'сохранена' if EXTRA_SAVED else 'не найдена'} ({EXTRA_MODEL_DIR})")

ner_model = None
if TRAIN_BASE:
    print("Базовая модель отсутствует — потребуется первоначальное обучение.")
    torch.set_num_threads(min(8, os.cpu_count() or 4))

    ner_model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(PERSON_LABELS),
        id2label=person_id2label,
        label2id=person_label2id
    )

    # Обучаем классификатор и последние два слоя RuBERT.
    for parameter in ner_model.bert.embeddings.parameters():
        parameter.requires_grad = False
    for layer in ner_model.bert.encoder.layer[:-2]:
        for parameter in layer.parameters():
            parameter.requires_grad = False

    trainable = sum(p.numel() for p in ner_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in ner_model.parameters())
    print("Всего параметров:", f"{total:,}")
    print("Обучаемых параметров:", f"{trainable:,}")
    print("Доля обучаемых:", f"{trainable / total:.2%}")
else:
    print("Базовое обучение пропущено: готовые веса уже есть.")

Базовая модель: сохранена (C:\Users\Vlad\1JupyterLab\CyberSec\rubert_pii_fio)
Дообученная модель: сохранена (C:\Users\Vlad\1JupyterLab\CyberSec\rubert_pii_fio_extra)
Базовое обучение пропущено: готовые веса уже есть.


In [27]:
def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, gold in zip(predictions, labels):
        pred_tags = []
        gold_tags = []

        for pred_id, gold_id in zip(prediction, gold):
            if gold_id == -100:
                continue

            pred_tags.append(
                person_id2label[int(pred_id)]
            )

            gold_tags.append(
                person_id2label[int(gold_id)]
            )

        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)

    return {
        "precision": ner_precision(
            true_labels, true_predictions,
            mode="strict", scheme=IOB2
        ),
        "recall": ner_recall(
            true_labels, true_predictions,
            mode="strict", scheme=IOB2
        ),
        "f1": ner_f1(
            true_labels, true_predictions,
            mode="strict", scheme=IOB2
        )
    }


In [28]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

training_args = TrainingArguments(
    output_dir="./rubert_fio_checkpoints",

    # Обучение (только если сохранённых весов нет)
    num_train_epochs=3,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Автоматический выбор CUDA/CPU при необходимости обучения
    use_cpu=(device.type == "cpu"),
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    logging_steps=25,
    logging_strategy="steps",
    report_to="none",
    seed=42
)

trainer = None
if TRAIN_BASE:
    trainer = Trainer(
        model=ner_model,
        args=training_args,
        train_dataset=ner_datasets["train"],
        eval_dataset=ner_datasets["validation"],
        data_collator=data_collator,
        compute_metrics=compute_ner_metrics
    )
    print("Trainer подготовлен: базовое обучение потребуется.")
else:
    print("Trainer для базового обучения не создаётся: модель сохранена.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer для базового обучения не создаётся: модель сохранена.


In [29]:
if TRAIN_BASE:
    train_result = trainer.train()
    print("\nБазовое обучение завершено.")
    print(train_result.metrics)
else:
    print("Базовое обучение пропущено: используются сохранённые веса.")

Базовое обучение пропущено: используются сохранённые веса.


In [30]:
if TRAIN_BASE:
    from seqeval.metrics import classification_report as ner_report

    test_output = trainer.predict(
        ner_datasets["test"],
        metric_key_prefix="test"
    )

    print("Общие метрики ФИО:")
    for key, value in test_output.metrics.items():
        if key in ("test_precision", "test_recall", "test_f1"):
            print(f"{key}: {value:.4f}")

    predictions = np.argmax(
        test_output.predictions, axis=-1
    )

    true_predictions = []
    true_labels = []

    for pred, gold in zip(
        predictions,
        test_output.label_ids
    ):
        pred_tags = []
        gold_tags = []

        for pred_id, gold_id in zip(pred, gold):
            if gold_id == -100:
                continue

            pred_tags.append(person_id2label[int(pred_id)])
            gold_tags.append(person_id2label[int(gold_id)])

        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)

    print("\nОтчёт по атрибутам:")
    print(
        ner_report(
            true_labels,
            true_predictions,
            mode="strict",
            scheme=IOB2,
            digits=4
        )
    )
else:
    print("Оценка промежуточной модели пропущена: ниже будет оценена итоговая RuBERT на factRuEval.")

Оценка промежуточной модели пропущена: ниже будет оценена итоговая RuBERT на factRuEval.


In [31]:
if TRAIN_BASE:
    trainer.save_model(str(BASE_MODEL_DIR))
    tokenizer.save_pretrained(str(BASE_MODEL_DIR))
    if not saved_rubert_exists(BASE_MODEL_DIR):
        raise RuntimeError("Не удалось сохранить базовую модель RuBERT")
    print("Базовая модель сохранена:", BASE_MODEL_DIR)
else:
    print("Базовая модель не перезаписывалась.")

# Освобождаем память перед возможным вторым этапом обучения.
if trainer is not None:
    del trainer
    trainer = None
if ner_model is not None:
    del ner_model
    ner_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Базовая модель не перезаписывалась.


In [32]:
# тесты по 9 атрибутам

import random
from collections import Counter

rng = random.Random(117)

def valid_number(length, validator, prefix=""):
    while True:
        value = prefix + "".join(
            str(rng.randrange(10))
            for _ in range(length - len(prefix))
        )
        if validator(value):
            return value


prefixes = {
    "EMAIL": " E-mail: ",
    "PHONE": " Телефон: ",
    "INN": " ИНН: ",
    "CARD": " Номер карты: ",
    "SNILS": " СНИЛС: ",
    "BIRTH_DATE": " Дата рождения: "
}

test_cases = []

for i in range(50):
    record = synthetic_bio["test"][i]
    text = record["text"]

    gold = [
        [span["label"].upper(), span["start"], span["end"]]
        for span in record["spans"]
    ]

    card = valid_number(16, valid_card, "4")
    snils = valid_number(11, valid_snils)

    values = {
        "EMAIL": f"test{i+1}@example.org",
        "PHONE": (
            f"+7 9{rng.randrange(100):02d} "
            f"{rng.randrange(1000):03d} "
            f"{rng.randrange(100):02d} "
            f"{rng.randrange(100):02d}"
        ),
        "INN": valid_number(12, valid_inn),
        "CARD": " ".join(card[j:j+4] for j in range(0, 16, 4)),
        "SNILS": f"{snils[:3]}-{snils[3:6]}-{snils[6:9]} {snils[9:]}",
        "BIRTH_DATE": (
            f"{rng.randint(1, 28):02d}."
            f"{rng.randint(1, 12):02d}."
            f"{rng.randint(1960, 2000)}"
        )
    }

    for label, value in values.items():
        text += prefixes[label]
        start = len(text)
        text += value
        gold.append([label, start, len(text)])

    test_cases.append((text, gold))


# Отрицательные примеры: неверные контрольные разряды
def corrupt(value):
    return value[:-1] + str((int(value[-1]) + 1) % 10)


for i in range(50):
    inn = corrupt(valid_number(12, valid_inn))
    card = corrupt(valid_number(16, valid_card, "4"))
    snils = corrupt(valid_number(11, valid_snils))

    text = (
        f"Проверка: ИНН {inn}; "
        f"карта {card}; "
        f"СНИЛС {snils}; "
        "дата рождения 31.02.2001."
    )
    test_cases.append((text, []))


counts = Counter(
    entity[0]
    for _, gold in test_cases
    for entity in gold
)

print("Всего тестовых текстов:", len(test_cases))
print("Количество эталонных сущностей:", dict(counts))

assert all(counts[label] >= 50 for label in ENTITY_TYPES)


Всего тестовых текстов: 100
Количество эталонных сущностей: {'SURNAME': 50, 'NAME': 50, 'PATRONYMIC': 50, 'EMAIL': 50, 'PHONE': 50, 'INN': 50, 'CARD': 50, 'SNILS': 50, 'BIRTH_DATE': 50}


In [33]:
if TRAIN_EXTRA:
    # Генерация дополнительных обучающих примеров со склонёнными именами и отчествами
    morph = pymorphy3.MorphAnalyzer()
    fake_fio = Faker("ru_RU")
    fake_fio.seed_instance(2027)
    rng = random.Random(2027)

    # Разные контексты и падежи
    CONTEXTS = [
        ("В документах указано: {NAME} {PATRONYMIC}.", "nomn"),
        ("По словам {NAME} {PATRONYMIC}, решение принято.", "gent"),
        ("Письмо передали {NAME} {PATRONYMIC}.", "datv"),
        ("Журналист говорил с {NAME} {PATRONYMIC}.", "ablt"),
        ("Речь шла о {NAME} {PATRONYMIC}.", "loct"),
        ("Встреча с {NAME} {PATRONYMIC} состоялась вчера.", "ablt"),
        ("Обсуждали кандидатуру {NAME} {PATRONYMIC}.", "gent")
    ]

    def inflect_fio(word, case, grammatical_tag):
        variants = morph.parse(word)
        parsed = next(
            (p for p in variants if grammatical_tag in p.tag),
            variants[0]
        )
        form = parsed.inflect({case})
        return form.word.capitalize() if form else word

    def generate_patronymic_example(i):
        gender = rng.choice(["male", "female"])
        template, case = rng.choice(CONTEXTS)

        name = getattr(fake_fio, f"first_name_{gender}")()
        patronymic = getattr(fake_fio, f"middle_name_{gender}")()

        name = inflect_fio(name, case, "Name")
        patronymic = inflect_fio(patronymic, case, "Patr")

        text = template.format(
            NAME=name,
            PATRONYMIC=patronymic
        )

        spans = []
        for label, value in [
            ("name", name),
            ("patronymic", patronymic)
        ]:
            start = text.index(value)
            spans.append({
                "label": label,
                "start": start,
                "end": start + len(value)
            })

        tokens = [
            {
                "text": m.group(),
                "start": m.start(),
                "end": m.end()
            }
            for m in WORD_RE.finditer(text)
        ]

        return {
            "id": f"extra_patronymic_{i}",
            "text": text,
            "tokens": tokens,
            "spans": spans
        }

    extra_bio = Dataset.from_list([
        generate_patronymic_example(i)
        for i in range(800)
    ]).map(create_bio_labels)

    print("Дополнительных текстов:", len(extra_bio))
    print("Пропущенных сущностей:", sum(extra_bio["bio_skipped"]))

    assert sum(extra_bio["bio_skipped"]) == 0

    for record in extra_bio.select(range(5)):
        print(record["text"])
else:
    print("Дополнительные обучающие примеры не генерируются: итоговая модель уже сохранена.")

Дополнительные обучающие примеры не генерируются: итоговая модель уже сохранена.


In [34]:
if TRAIN_EXTRA:
    extra_windows = Dataset.from_list([
        window
        for record in extra_bio
        for window in prepare_ner_windows(record)
    ])

    ner_datasets["train"] = concatenate_datasets([
        base_train_windows,
        extra_windows
    ])

    print("Обучающих последовательностей:", len(ner_datasets["train"]))
else:
    print("Подготовка дополнительных обучающих окон пропущена.")

Подготовка дополнительных обучающих окон пропущена.


In [35]:
from dataclasses import replace

if TRAIN_EXTRA:
    if not saved_rubert_exists(BASE_MODEL_DIR):
        raise FileNotFoundError(
            f"Для дообучения нужна базовая модель: {BASE_MODEL_DIR}"
        )

    more_model = AutoModelForTokenClassification.from_pretrained(str(BASE_MODEL_DIR))
    more_args = replace(
        training_args,
        output_dir="./rubert_pii_fio_extra_checkpoints",
        learning_rate=1e-5,
        num_train_epochs=2
    )

    more_trainer = Trainer(
        model=more_model,
        args=more_args,
        train_dataset=ner_datasets["train"],
        eval_dataset=ner_datasets["validation"],
        data_collator=data_collator,
        compute_metrics=compute_ner_metrics
    )

    print("Итоговой модели нет — запускается дополнительное обучение.")
    more_trainer.train()
    more_trainer.save_model(str(EXTRA_MODEL_DIR))
    tokenizer.save_pretrained(str(EXTRA_MODEL_DIR))

    if not saved_rubert_exists(EXTRA_MODEL_DIR):
        raise RuntimeError("Не удалось сохранить итоговую модель RuBERT")
    print("Итоговая модель сохранена:", EXTRA_MODEL_DIR)

    del more_trainer, more_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Дополнительное обучение пропущено: итоговая модель уже сохранена.")

Дополнительное обучение пропущено: итоговая модель уже сохранена.


In [36]:
# Итоговая модель RuBERT
# Морфологический анализ нужен уже в тестах распознавания ниже.
morph = pymorphy3.MorphAnalyzer()
if not saved_rubert_exists(EXTRA_MODEL_DIR):
    raise FileNotFoundError(f"Итоговая модель отсутствует: {EXTRA_MODEL_DIR}")

new_tokenizer = AutoTokenizer.from_pretrained(str(EXTRA_MODEL_DIR), use_fast=True)
new_tokenizer.model_max_length = 128

new_model = AutoModelForTokenClassification.from_pretrained(
    str(EXTRA_MODEL_DIR)
)

new_ner = pipeline(
    "token-classification",
    model=new_model,
    tokenizer=new_tokenizer,
    aggregation_strategy="first",
    stride=32,
    device=0 if torch.cuda.is_available() else -1
)

# Распознавание только ФИО

# Обращения и титулы, не являющиеся именами
PERSON_TITLES = {
    "миссис", "мисс", "мистер",
    "господин", "госпожа",
    "мадам", "сэр"
}

def detect_person(text):
    entities = []

    for entity in new_ner(text):
        label = entity["entity_group"]
        start = entity["start"]
        end = entity["end"]

        value = text[start:end].strip()
        if not any(ch.isalpha() for ch in value):
            continue
        normalized = value.lower()
        # Отсекаем очевидные нарицательные существительные,
        # ошибочно распознанные как NAME или SURNAME.
        if (
            label in {"NAME", "SURNAME"}
            and re.fullmatch(r"[а-яё]+", value)
        ):
            parses = morph.parse(value)
        
            if (
                any("NOUN" in p.tag for p in parses)
                and not any(
                    "Name" in p.tag or "Surn" in p.tag
                    for p in parses
                )
            ):
                continue
        # Исключаем обращения из распознанных ФИО
        if normalized in PERSON_TITLES:
            continue

        # Отсекаем конкретное ложное срабатывание
        # Но разрешаем «Цыган» с заглавной буквы
        # как потенциальную фамилию
        if label == "NAME" and value == "цыган":
            continue

        entities.append([label, start, end])

    return entities


# Распознавание всех ПДн
def detect_pii(text):
    if not isinstance(text, str):
        raise TypeError("text должен быть строкой")

    result = detect_structured(text)

    for label, start, end in detect_person(text):
        if not any(start < e and s < end for _, s, e in result):
            result.append([label, start, end])

    return sorted(result, key=lambda x: x[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [37]:
# f1-мера по BIO

import re
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2


def to_bio(text, entities):
    tokens = list(re.finditer(r"\w+|[^\w\s]", text))
    tags = ["O"] * len(tokens)

    for label, start, end in sorted(entities, key=lambda x: x[1]):
        indices = [
            i for i, token in enumerate(tokens)
            if token.start() < end and token.end() > start
        ]

        if indices and all(tags[i] == "O" for i in indices):
            for j, i in enumerate(indices):
                tags[i] = ("B-" if j == 0 else "I-") + label

    return tags

y_true, y_pred = [], []

for text, gold in test_cases:
    y_true.append(to_bio(text, gold))
    y_pred.append(to_bio(text, detect_pii(text)))

report = classification_report(
    y_true, y_pred,
    mode="strict",
    scheme=IOB2,
    digits=4,
    output_dict=True
)

print("Общая weighted F1:", round(report["weighted avg"]["f1-score"], 4))

print(classification_report(
    y_true, y_pred,
    mode="strict",
    scheme=IOB2,
    digits=4
))


Общая weighted F1: 0.9989
              precision    recall  f1-score   support

  BIRTH_DATE     1.0000    1.0000    1.0000        50
        CARD     1.0000    1.0000    1.0000        50
       EMAIL     1.0000    1.0000    1.0000        50
         INN     1.0000    1.0000    1.0000        50
        NAME     1.0000    1.0000    1.0000        50
  PATRONYMIC     1.0000    1.0000    1.0000        50
       PHONE     0.9804    1.0000    0.9901        50
       SNILS     1.0000    1.0000    1.0000        50
     SURNAME     1.0000    1.0000    1.0000        50

   micro avg     0.9978    1.0000    0.9989       450
   macro avg     0.9978    1.0000    0.9989       450
weighted avg     0.9978    1.0000    0.9989       450



In [38]:
# тесты на testset

real_true, real_pred = [], []

for record in bio_datasets["test"]:
    text = record["text"]

    gold = [
        [s["label"].upper(), s["start"], s["end"]]
        for s in record["spans"]
        if s["label"].lower() in PERSON_TYPES
    ]

    real_true.append(to_bio(text, gold))
    real_pred.append(to_bio(text, detect_person(text)))

print(classification_report(
    real_true, real_pred,
    mode="strict",
    scheme=IOB2,
    digits=4
))


              precision    recall  f1-score   support

        NAME     0.9539    0.8951    0.9236       810
  PATRONYMIC     0.7188    0.8519    0.7797        27
     SURNAME     0.9669    0.8754    0.9189      1268

   micro avg     0.9577    0.8827    0.9187      2105
   macro avg     0.8799    0.8741    0.8740      2105
weighted avg     0.9587    0.8827    0.9189      2105



In [39]:
import re

count = 0

for doc in bio_datasets["test"]:
    text = doc["text"]

    gold = to_bio(text, [
        [s["label"].upper(), s["start"], s["end"]]
        for s in doc["spans"]
        if s["label"].lower() in PERSON_TYPES
    ])

    # Предсказания новой модели
    entities = new_ner(text)

    pred = to_bio(text, [
        [e["entity_group"], e["start"], e["end"]]
        for e in entities
    ])

    tokens = list(re.finditer(r"\w+|[^\w\s]", text))

    for i, tag in enumerate(gold):
        if tag == "B-PATRONYMIC" and pred[i] != tag:
            m = tokens[i]
            start, end = m.start(), m.end()

            # Выделяем ошибочное слово в контексте
            context = (
                text[max(0, start - 60):start]
                + " >>>" + text[start:end] + "<<< "
                + text[end:min(len(text), end + 60)]
            )

            print("Отчество:", text[start:end])
            print("Позиция в тексте:", start, end)
            print("Контекст:", context.replace("\n", " "))
            print("Предсказанная BIO-метка:", pred[i])

            # Что именно модель распознала на этом месте
            found = [
                e for e in entities
                if e["start"] < end and e["end"] > start
            ]

            if found:
                for e in found:
                    print(
                        "Модель определила:",
                        repr(text[e["start"]:e["end"]]),
                        "как", e["entity_group"],
                        f"на позиции {e['start']}:{e['end']}"
                    )
            else:
                print("Модель ничего не обнаружила на этом месте")

            print("-" * 60)

            count += 1
            if count >= 10:
                break

    if count >= 10:
        break

print("Найдено ошибок:", count)

Отчество: Альфонса
Позиция в тексте: 10949 10957
Контекст: лах и фешенебельных отелях — для королей Густава Шведского,  >>>Альфонса<<<  Испанского, принца Уэльского, для Вандербильтов и Ротшильдо
Предсказанная BIO-метка: B-NAME
Модель определила: 'Альфонса' как NAME на позиции 10949:10957
------------------------------------------------------------
Отчество: С
Позиция в тексте: 1724 1725
Контекст: ждения проблем демократизации нашей общей страны, начатой М. >>>С<<< . Горбачевым.  Впрочем, чуть ранее та же самая история случи
Предсказанная BIO-метка: O
Модель ничего не обнаружила на этом месте
------------------------------------------------------------
Отчество: В
Позиция в тексте: 2001 2002
Контекст: ия штатов в Московском государственном университете имени М. >>>В<<< . Ломоносова. Тогда произошла волна увольнений сотрудников л
Предсказанная BIO-метка: O
Модель ничего не обнаружила на этом месте
------------------------------------------------------------
Отчество: А
Позиция в тексте:

In [40]:
# Не прерываем Run All, если пользовательский input.txt ещё не создан.
file_path = Path("input.txt")
if file_path.is_file():
    text = file_path.read_text(encoding="utf-8-sig")
    result = detect_pii(text)
    print(result)
    print("\nНайденные персональные данные:")
    for label, start, end in result:
        print(f"{label}: {text[start:end]!r} [{start}, {end}]")
else:
    print("input.txt отсутствует — проверка пользовательского файла пропущена.")

[['SURNAME', 118, 124], ['NAME', 125, 132], ['PATRONYMIC', 133, 142], ['BIRTH_DATE', 179, 189], ['PHONE', 206, 222], ['EMAIL', 249, 274], ['INN', 306, 318], ['SNILS', 326, 340], ['CARD', 366, 385], ['SURNAME', 517, 526], ['NAME', 527, 532], ['PATRONYMIC', 533, 541], ['EMAIL', 610, 639], ['PHONE', 652, 668], ['BIRTH_DATE', 684, 694], ['INN', 700, 712], ['SNILS', 720, 734], ['CARD', 776, 795], ['NAME', 797, 802], ['PATRONYMIC', 803, 811], ['SURNAME', 933, 940], ['NAME', 941, 948], ['PATRONYMIC', 949, 958], ['NAME', 989, 994], ['PATRONYMIC', 995, 1005], ['SURNAME', 1006, 1013]]

Найденные персональные данные:
SURNAME: 'Иванов' [118, 124]
NAME: 'Алексей' [125, 132]
PATRONYMIC: 'Сергеевич' [133, 142]
BIRTH_DATE: '14.03.1992' [179, 189]
PHONE: '+7 999 123-45-67' [206, 222]
EMAIL: 'alexey.ivanov@example.org' [249, 274]
INN: '128860301876' [306, 318]
SNILS: '186-008-443 69' [326, 340]
CARD: '4111 1111 1111 1111' [366, 385]
SURNAME: 'Кузнецова' [517, 526]
NAME: 'Мария' [527, 532]
PATRONYMIC: 'П

# лр2

In [41]:
# ЛР2: необходимые объекты без запуска обучения RuBERT
import re
import string
import pymorphy3
from faker import Faker
from random import SystemRandom

PERSON_TAGS = {"NAME": "Name", "SURNAME": "Surn"}
morph = pymorphy3.MorphAnalyzer()
anon_fake = Faker("ru_RU")
anon_rng = SystemRandom()


In [42]:

# Фамилии, не изменяющиеся по падежам
INDECLINABLE_SURNAMES = [
    "Гюго", "Дюма", "Верди", "Моро", "Россини", "Моне"
]

INITIAL_LETTERS = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЭЮЯ"


def restore_person_case(original, replacement):
    if original.isupper():
        return replacement.upper()
    if original.islower():
        return replacement.lower()
    if original.istitle():
        return replacement.capitalize()
    return replacement


def replace_person_base(value, label, replacements, left_context=""):
    if label not in PERSON_TAGS or not value:
        raise ValueError(f"Некорректный атрибут: {label}, {value!r}")

    # 1. Инициалы: М -> К, М. -> К.
    initial = re.fullmatch(r"([А-Яа-яЁё])(\.?)", value)

    if initial:
        letter, dot = initial.groups()
        key = (label, "INITIAL", letter.upper())

        if key not in replacements:
            replacements[key] = anon_rng.choice([
                c for c in INITIAL_LETTERS
                if c != letter.upper()
            ])

        return restore_person_case(
            letter, replacements[key]
        ) + dot

    # 2. Обычный морфологический анализ
    tag = PERSON_TAGS[label]
    parses = [
        p for p in morph.parse(value)
        if tag in p.tag
        and p.tag.gender in ("masc", "femn")
    ]

    # Запасной анализ редких имён:
    # Робера, Фауста, Эдисона и т.п.
    if not parses and label == "NAME":
        parses = [
            p for p in morph.parse(value)
            if "NOUN" in p.tag
            and "anim" in p.tag
            and p.tag.gender in ("masc", "femn")
            and p.tag.case is not None
        ]

    if parses:
        original = parses[0]
        gender = original.tag.gender
        case = original.tag.case
        number = original.tag.number
        normal_form = original.normal_form

    elif label == "SURNAME":
        # 3. Неизвестная склоняемая фамилия:
        # Брохесом -> предположительно мужской, тв. падеж
        if value.lower().endswith(("ом", "ем")):
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = value[:-2].lower()

        else:
            # 4. Неоднозначная или несклоняемая фамилия.
            # Используем другую несклоняемую фамилию:
            # Мураками -> Верди, Моне -> Гюго.
            key = ("SURNAME", "INDECL", value.lower())

            if key not in replacements:
                candidates = [
                    s for s in INDECLINABLE_SURNAMES
                    if s.lower() != value.lower()
                ]
                replacements[key] = anon_rng.choice(candidates)

            return restore_person_case(
                value, replacements[key]
            )


    elif label == "NAME":
        lower = value.lower()

        if lower == "лизка":
            gender = "femn"
            case = "nomn"
            number = "sing"
            normal_form = "лизка"

        elif (
            lower == "мая"
            and re.search(r"\bмальчика\s+$", left_context.lower())
        ):
            gender = "masc"
            case = "gent"
            number = "sing"
            normal_form = "май"

        elif lower.endswith(("ом", "ем")):
            # Орсоном -> Орсон, творительный падеж
            gender = "masc"
            case = "ablt"
            number = "sing"
            normal_form = lower[:-2]
            
        elif lower.endswith(("ей", "ий", "й")):
            # Кащей -> Кащей, предполагаемый именительный
            gender = "masc"
            case = "nomn"
            number = "sing"
            normal_form = lower

        else:
            raise ValueError(
                f"Неизвестная форма NAME: {value}"
            )

    # 5. Согласование повторов по нормальной форме
    key = (label, normal_form, gender)
    grammemes = {g for g in (case, number) if g}

    if key not in replacements:
        sex = "male" if gender == "masc" else "female"
        method = "first_name" if label == "NAME" else "last_name"

        for _ in range(100):
            candidate = getattr(
                anon_fake, f"{method}_{sex}"
            )()

            options = [
                p for p in morph.parse(candidate)
                if tag in p.tag and p.tag.gender == gender
            ]

            if not options:
                continue

            candidate_parse = options[0]

            if candidate_parse.normal_form == normal_form:
                continue

            if grammemes and candidate_parse.inflect(grammemes) is None:
                continue

            replacements[key] = candidate_parse
            break
        else:
            raise ValueError(
                f"Не удалось подобрать замену: {value}"
            )

    # 6. Склонение выбранной замены
    replacement = replacements[key]
    
    inflected = (
        replacement.inflect(grammemes)
        if grammemes else replacement
    )

    if inflected is None:
        raise ValueError(f"Ошибка склонения: {value}")

    return restore_person_case(value, inflected.word)

In [43]:

# Расширение базового алгоритма: уменьшительные имена и латиница

latin_fake = Faker("en_US")

LATIN_FEMALE = {"bonnie"}
LATIN_MALE = {"george", "walt"}


def replace_person(value, label, replacements, left_context=""):
    # Неоднозначное по роду имя: Саша
    if label == "NAME" and value.casefold() == "саша":
        key = ("NAME", "NICKNAME", "саша")

        if key not in replacements:
            # Женя и Валя могут быть мужскими и женскими именами
            replacements[key] = anon_rng.choice(["Женя", "Валя"])

        return restore_person_case(value, replacements[key])

    # Латинский инициал: H -> R
    if label == "NAME" and re.fullmatch(r"[A-Za-z]", value):
        key = ("NAME", "LATIN_INITIAL", value.upper())

        if key not in replacements:
            letters = string.ascii_uppercase.replace(value.upper(), "")
            replacements[key] = anon_rng.choice(letters)

        return restore_person_case(value, replacements[key])

    # Латинские имена: George -> Michael
    if label == "NAME" and re.fullmatch(r"[A-Za-z]+", value):
        key = ("NAME", "LATIN", value.casefold())

        if key not in replacements:
            name = value.casefold()

            if name in LATIN_FEMALE:
                generate = latin_fake.first_name_female
            elif name in LATIN_MALE:
                generate = latin_fake.first_name_male
            else:
                # Пол неизвестен: нейтральные варианты
                generate = lambda: anon_rng.choice(
                    ["Alex", "Taylor", "Jordan", "Casey"]
                )

            for _ in range(100):
                candidate = generate()

                if candidate.casefold() != name:
                    replacements[key] = candidate
                    break
            else:
                raise ValueError(f"Не удалось заменить NAME: {value}")

        return restore_person_case(value, replacements[key])

    # Все обычные кириллические имена и фамилии
    return replace_person_base(
        value, label, replacements, left_context
    )


In [44]:

def anon_people(text, entities):
    replacements = {}
    result = text

    # Проверяем координаты и оставляем только имена и фамилии
    selected = []

    for label, start, end in entities:
        if label not in PERSON_TAGS:
            continue

        if not (0 <= start < end <= len(text)):
            raise ValueError("Некорректные координаты ПДн")

        selected.append((label, start, end))

    selected.sort(key=lambda x: x[1])

    # Не допускаем пересечения сущностей
    for a, b in zip(selected, selected[1:]):
        if a[2] > b[1]:
            raise ValueError("Пересекающиеся сущности ПДн")

    # Заменяем справа налево, чтобы не сбить индексы
    for label, start, end in reversed(selected):
        old_value = text[start:end]

        new_value = replace_person(
            old_value, label, replacements
        )

        result = result[:start] + new_value + result[end:]

    return result


In [45]:

import re
import string
from random import SystemRandom

anon_rng = SystemRandom()


def random_other(char, alphabet):
    """Случайный символ, обязательно отличный от исходного."""
    return anon_rng.choice(alphabet.replace(char, ""))


def replace_email(value):
    """Меняет часть до @, сохраняя домен и оформление."""
    parts = value.split("@")

    if len(parts) != 2:
        raise ValueError("Некорректный EMAIL")

    local, domain = parts

    # Проверка некорректного расположения точек и дефисов
    if (
        local.startswith(".")
        or local.endswith(".")
        or ".." in local
        or ".." in domain
        or any(
            part.startswith("-") or part.endswith("-")
            for part in domain.split(".")
        )
    ):
        raise ValueError(f"Некорректный EMAIL: {value}")

    if not re.fullmatch(r"[A-Za-z0-9._%+-]+", local):
        raise ValueError("Некорректная локальная часть EMAIL")

    if not re.fullmatch(
        r"[A-Za-zА-Яа-яЁё0-9-]+(?:\.[A-Za-zА-Яа-яЁё0-9-]+)+",
        domain
    ):
        raise ValueError("Некорректный домен EMAIL")

    result = []

    for char in local:
        if char in string.ascii_lowercase:
            result.append(random_other(char, string.ascii_lowercase))
        elif char in string.ascii_uppercase:
            result.append(random_other(char, string.ascii_uppercase))
        elif char in string.digits:
            result.append(random_other(char, string.digits))
        else:
            result.append(char)

    return "".join(result) + "@" + domain


def replace_phone(value):
    """Сохраняет первые 4 цифры и все нецифровые символы."""
    digits = [c for c in value if c in string.digits]

    if not 10 <= len(digits) <= 15:
        raise ValueError("Некорректная длина PHONE")

    if any(c not in string.digits + "+-(). \t" for c in value):
        raise ValueError("Недопустимые символы в PHONE")

    result = []
    digit_index = 0

    for char in value:
        if char in string.digits:
            if digit_index < 4:
                result.append(char)
            else:
                result.append(random_other(char, string.digits))
            digit_index += 1
        else:
            result.append(char)

    return "".join(result)


In [46]:

email = "ivan.petrov1988@gmail.com"
phone = "+7 (999) 123-45-67"

new_email = replace_email(email)
new_phone = replace_phone(phone)

print("EMAIL:")
print("Исходный:     ", email)
print("Обезличенный:", new_email)

print("\nPHONE:")
print("Исходный:     ", phone)
print("Обезличенный:", new_phone)

# Проверка требований
assert new_email != email
assert new_email.split("@")[1] == email.split("@")[1]

old_digits = re.findall(r"[0-9]", phone)
new_digits = re.findall(r"[0-9]", new_phone)

assert old_digits[:4] == new_digits[:4]
assert old_digits != new_digits
assert len(phone) == len(new_phone)

assert [
    (i, c) for i, c in enumerate(phone) if c not in string.digits
] == [
    (i, c) for i, c in enumerate(new_phone) if c not in string.digits
]

# Проверяем случайность
assert replace_email(email) != replace_email(email)
assert replace_phone(phone) != replace_phone(phone)

print("\nВсе проверки пройдены.")


EMAIL:
Исходный:      ivan.petrov1988@gmail.com
Обезличенный: hdhr.clechy7797@gmail.com

PHONE:
Исходный:      +7 (999) 123-45-67
Обезличенный: +7 (999) 708-23-08

Все проверки пройдены.


In [47]:

def replace_inn(value):
    """Сохраняет первые 4 цифры и пересчитывает контрольные разряды."""

    if not valid_inn(value):
        raise ValueError(f"Некорректный ИНН: {value}")

    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)

    while True:
        first10 = value[:4] + "".join(
            str(anon_rng.randrange(10)) for _ in range(6)
        )

        numbers = list(map(int, first10))

        c11 = sum(
            n * w for n, w in zip(numbers, w11)
        ) % 11 % 10

        c12 = sum(
            n * w for n, w in zip(numbers + [c11], w12)
        ) % 11 % 10

        result = first10 + str(c11) + str(c12)

        if result != value:
            return result


In [48]:

def replace_card(value):
    """Сохраняет первые 6 и последние 4 цифры, формат и Luhn."""

    if (
        not re.fullmatch(r"[0-9]+(?:[ \t-][0-9]+)*", value)
        or not valid_card(value)
    ):
        raise ValueError(f"Некорректная банковская карта: {value}")

    original = digits_only(value)

    for _ in range(1000):
        middle = "".join(
            str(anon_rng.randrange(10))
            for _ in range(len(original) - 10)
        )

        candidate = original[:6] + middle + original[-4:]

        if candidate != original and valid_luhn(candidate):
            digits_iter = iter(candidate)

            return "".join(
                next(digits_iter) if char in "0123456789" else char
                for char in value
            )

    raise RuntimeError("Не удалось сгенерировать валидную карту")


In [49]:

ANON_TYPES = {"NAME", "SURNAME", "EMAIL", "INN", "PHONE", "CARD"}


def restore_digits(original, new_digits):
    """Возвращает новые цифры в исходное оформление."""
    digits = iter(new_digits)
    return "".join(
        next(digits) if ch in "0123456789" else ch
        for ch in original
    )


def anon(text, entities):
    """Синонимически обезличивает ПДн по координатам из ЛР1."""

    if not isinstance(text, str):
        raise TypeError("Текст должен быть строкой")

    # Допускаем как один атрибут, так и список атрибутов
    if isinstance(entities, (list, tuple)) and len(entities) == 3 and isinstance(entities[0], str):
        entities = [entities]

    spans = []

    for entity in entities:
        if len(entity) != 3:
            raise ValueError(f"Некорректная сущность: {entity}")

        label, start, end = entity

        if not (isinstance(start, int) and isinstance(end, int)
                and 0 <= start < end <= len(text)):
            raise ValueError(f"Некорректные координаты: {entity}")

        if label in ANON_TYPES:
            spans.append((label, start, end))

    spans.sort(key=lambda x: x[1])

    for left, right in zip(spans, spans[1:]):
        if left[2] > right[1]:
            raise ValueError("Обнаружены пересекающиеся сущности")

    person_cache = {}
    structured_cache = {}
    result = text

    for label, start, end in reversed(spans):
        original = text[start:end]

        if label in ("NAME", "SURNAME"):
            replacement = replace_person(
                original,
                label,
                person_cache,
                left_context=text[max(0, start - 80):start]
            )

        elif label == "EMAIL":
            key = (label, original)

            if key not in structured_cache:
                structured_cache[key] = replace_email(original)

            replacement = structured_cache[key]

        else:
            # Нормализуем числовой атрибут
            normalized = digits_only(original)
            key = (label, normalized)

            # Проверяем каждое входное значение
            if label == "INN" and not valid_inn(original):
                raise ValueError(f"Некорректный ИНН: {original}")

            if label == "CARD" and not valid_card(original):
                raise ValueError(f"Некорректная карта: {original}")

            if label == "PHONE":
                if (
                    not 10 <= len(normalized) <= 15
                    or any(c not in "0123456789+-(). \t" for c in original)
                ):
                    raise ValueError(f"Некорректный телефон: {original}")

            if key not in structured_cache:
                generators = {
                    "INN": replace_inn,
                    "PHONE": replace_phone,
                    "CARD": replace_card
                }

                generated = generators[label](original)
                structured_cache[key] = digits_only(generated)

            replacement = restore_digits(
                original, structured_cache[key]
            )

        if replacement == original:
            raise ValueError(
                f"Значение не изменилось: {label} = {original}"
            )

        # Заменяем справа налево
        result = result[:start] + replacement + result[end:]

    return result


In [50]:

inn = "770698959438"
card = "4561 2612 1234 5467"

new_inn = replace_inn(inn)
new_card = replace_card(card)

print("ИНН:")
print("Исходный:     ", inn)
print("Обезличенный:", new_inn)
print("Валидность:  ", valid_inn(new_inn))

print("\nCARD:")
print("Исходная:    ", card)
print("Обезличенная:", new_card)
print("Валидность:  ", valid_card(new_card))

# Проверяем ИНН
assert new_inn[:4] == inn[:4]
assert len(new_inn) == 12
assert new_inn != inn
assert valid_inn(new_inn)

# Проверяем банковскую карту
old_digits = digits_only(card)
new_digits = digits_only(new_card)

assert new_digits[:6] == old_digits[:6]
assert new_digits[-4:] == old_digits[-4:]
assert new_digits != old_digits
assert valid_card(new_card)

# Проверяем, что оформление полностью сохранилось
assert len(new_card) == len(card)
assert all(
    a == b
    for a, b in zip(card, new_card)
    if a not in "0123456789"
)

# Проверяем случайность нескольких запусков
assert len({replace_inn(inn) for _ in range(5)}) > 1
assert len({replace_card(card) for _ in range(5)}) > 1

# Проверяем реакцию на ошибочные данные
for func, invalid in [
    (replace_inn, "770698959430"),
    (replace_inn, "123"),
    (replace_card, "4111 1111 1111 1112"),
    (replace_card, "**** 5467"),
]:
    try:
        func(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Не отклонено: {invalid}")

print("\nВсе проверки пройдены.")


ИНН:
Исходный:      770698959438
Обезличенный: 770606228078
Валидность:   True

CARD:
Исходная:     4561 2612 1234 5467
Обезличенная: 4561 2628 2333 5467
Валидность:   True

Все проверки пройдены.


In [51]:

text = (
    "Иванов Иван написал с ivan.petrov1988@gmail.com, "
    "указал ИНН 770698959438, телефон +7 (999) 123-45-67 "
    "и карту 4561 2612 1234 5467. "
    "Позднее с Ивановым Иваном связались повторно."
)

samples = [
    ("SURNAME", "Иванов"),
    ("NAME", "Иван"),
    ("EMAIL", "ivan.petrov1988@gmail.com"),
    ("INN", "770698959438"),
    ("PHONE", "+7 (999) 123-45-67"),
    ("CARD", "4561 2612 1234 5467"),
    ("SURNAME", "Ивановым"),
    ("NAME", "Иваном")
]

entities = []

for label, value in samples:
    match = re.search(
        rf"(?<!\w){re.escape(value)}(?!\w)", text
    )
    assert match is not None, f"Не найдено: {value}"
    entities.append([label, match.start(), match.end()])

first = anon(text, entities)
second = anon(text, entities)

print("ИСХОДНЫЙ ТЕКСТ:")
print(text)

print("\nОБЕЗЛИЧЕННЫЙ ТЕКСТ:")
print(first)

print("\nПОВТОРНЫЙ ЗАПУСК:")
print(second)

assert first != text
assert first != second

print("\nПроверка пройдена.")


ИСХОДНЫЙ ТЕКСТ:
Иванов Иван написал с ivan.petrov1988@gmail.com, указал ИНН 770698959438, телефон +7 (999) 123-45-67 и карту 4561 2612 1234 5467. Позднее с Ивановым Иваном связались повторно.

ОБЕЗЛИЧЕННЫЙ ТЕКСТ:
Дементьев Симон написал с hmrz.oaojpp0564@gmail.com, указал ИНН 770667232160, телефон +7 (999) 561-73-46 и карту 4561 2624 4226 5467. Позднее с Дементьевым Симоном связались повторно.

ПОВТОРНЫЙ ЗАПУСК:
Князев Всемил написал с xbjv.cnvxxf4834@gmail.com, указал ИНН 770640819585, телефон +7 (999) 961-24-20 и карту 4561 2621 2546 5467. Позднее с Князевым Всемилом связались повторно.

Проверка пройдена.


In [52]:

# Расширенная проверка ЛР2

inn = "770698959438"
card = "4561 2612 1234 5467"

# 1. Стоимость ошибки: контрольные разряды
for _ in range(200):
    new_inn = replace_inn(inn)
    new_card = replace_card(card)

    assert valid_inn(new_inn)
    assert new_inn[:4] == inn[:4]
    assert new_inn != inn

    assert valid_card(new_card)
    assert digits_only(new_card)[:6] == digits_only(card)[:6]
    assert digits_only(new_card)[-4:] == digits_only(card)[-4:]
    assert new_card != card

print("✓ 200 генераций ИНН и CARD прошли проверки")

# 2. Одинаковый EMAIL в одном тексте
email = "ivan.petrov@gmail.com"
text = f"Письмо от {email}. Ответ направлен на {email}."

entities = [
    ["EMAIL", m.start(), m.end()]
    for m in re.finditer(re.escape(email), text)
]

result = anon(text, entities)
emails = re.findall(r"[A-Za-z0-9._%+-]+@gmail\.com", result)

assert len(emails) == 2
assert emails[0] == emails[1]
assert emails[0] != email

print("✓ Повторяющиеся EMAIL заменяются согласованно")

# 3. Один телефон в разных форматах
text = "+7 (999) 123-45-67 / +79991234567"
entities = [
    ["PHONE", 0, 18],
    ["PHONE", 21, len(text)]
]

# Координаты определяем автоматически, чтобы исключить ошибку
entities = [
    ["PHONE", m.start(), m.end()]
    for m in re.finditer(r"\+7(?: \(999\) 123-45-67|9991234567)", text)
]

result = anon(text, entities)
left, right = result.split(" / ")

assert digits_only(left) == digits_only(right)

print("✓ Телефоны в разных форматах получают одну замену")

# 4. Некорректные входные значения
bad_cases = [
    ("123", [["INN", 0, 3]]),
    ("abc", [["EMAIL", 0, 3]]),
    ("abc", [["NAME", 0, 10]]),
    ("abc", [["NAME", 0, 2], ["NAME", 1, 3]])
]

for text, entities in bad_cases:
    try:
        anon(text, entities)
    except (ValueError, TypeError):
        pass
    else:
        raise AssertionError(
            f"Не обработан ошибочный ввод: {text}, {entities}"
        )

print("✓ Некорректные значения отклоняются")
print("\nВсе расширенные проверки пройдены.")


✓ 200 генераций ИНН и CARD прошли проверки
✓ Повторяющиеся EMAIL заменяются согласованно
✓ Телефоны в разных форматах получают одну замену
✓ Некорректные значения отклоняются

Все расширенные проверки пройдены.


In [53]:

from collections import Counter

TEST_LIMIT = 133
SUPPORTED = {"NAME", "SURNAME", "EMAIL", "INN", "PHONE", "CARD"}

stats = Counter()
errors = []
shown = 0

for doc in datasets["test"]:
    text = doc["text"]

    # Эталонные координаты имён и фамилий из factRuEval
    gold = [
        [s["label"].upper(), s["start"], s["end"]]
        for s in doc["spans"]
        if s["label"].lower() in {"name", "surname"}
    ]

    if not gold:
        continue

    stats["documents"] += 1

    # 1. Проверка обезличивания по эталонной разметке
    try:
        gold_result = anon(text, gold)
        assert gold_result != text
        stats["gold_success"] += 1
    except Exception as exc:
        stats["gold_errors"] += 1
        errors.append((doc["id"], "GOLD", str(exc)))

    # 2. Полная цепочка: RuBERT + правила -> anon()
    try:
        entities = [
            e for e in detect_pii(text)
            if e[0] in SUPPORTED
        ]

        if not entities:
            stats["no_entities"] += 1
            continue

        result = anon(text, entities)
        assert result != text

        stats["pipeline_success"] += 1
        stats["replaced_entities"] += len(entities)

        # Показываем результат именно этого вызова anon(), без повторной генерации.
        if shown < 3:
            print(f"\nДокумент: {doc['id']}")
            print("БЫЛО:")
            print(text[:500].replace("\n", " "))
            print("\nСТАЛО:")
            print(result[:500].replace("\n", " "))
            print("-" * 65)
            shown += 1

    except Exception as exc:
        stats["pipeline_errors"] += 1
        errors.append((doc["id"], "PIPELINE", str(exc)))

    if stats["documents"] >= TEST_LIMIT:
        break

print("\nИТОГИ ПРОВЕРКИ")
for key, value in stats.items():
    print(f"{key}: {value}")

print("\nОШИБКИ:")
if errors:
    for item in errors[:10]:
        print(item)
else:
    print("Не обнаружены")



Документ: book_3539
БЫЛО:
Александр Вертинский. «Я не знаю, зачем и кому это нужно…»  21 марта 1889 года родился главный русский шансонье XX века, печальный Пьеро, вписавший свою судьбу в историю отечественной культуры  Жизнь с самого начала оставляла для Александра Вертинского слишком много вопросов без ответов. Слишком много «пустого» пространства. И он научился заполнять его вымыслом. Создал собственный театр с безумным множеством персонажей, каждый из которых — от сироток-калек и безымянных кокаинеточек до гениальных

СТАЛО:
Максим Сысоев. «Я не знаю, зачем и кому это нужно…»  21 марта 1889 года родился главный русский шансонье XX века, печальный Пьеро, вписавший свою судьбу в историю отечественной культуры  Жизнь с самого начала оставляла для Максима Сысоева слишком много вопросов без ответов. Слишком много «пустого» пространства. И он научился заполнять его вымыслом. Создал собственный театр с безумным множеством персонажей, каждый из которых — от сироток-калек и безымянных кока

In [54]:

for doc_id in ["book_3539", "book_3677"]:
    doc = next(
        d for d in datasets["test"]
        if d["id"] == doc_id
    )

    text = doc["text"]
    entities = detect_person(text)

    print(f"\nДокумент: {doc_id}")
    print("Найденные ФИО:")

    for label, start, end in entities:
        print(f"{label:12} {text[start:end]}")



Документ: book_3539
Найденные ФИО:
NAME         Александр
SURNAME      Вертинский
NAME         Александра
SURNAME      Вертинского
SURNAME      Вертинского
NAME         Георгий
SURNAME      Иванов
NAME         Георгия
SURNAME      Иванова
NAME         Георгий
SURNAME      Иванов
SURNAME      Вертинского
NAME         Саша
NAME         Лизка
NAME         Надя
NAME         Надю
NAME         Саша
NAME         Александр
NAME         Н
PATRONYMIC   Н
SURNAME      Вертинской
SURNAME      Вертинскому
NAME         Надя
SURNAME      Вертинского
NAME         Надежда
PATRONYMIC   Николаевна
NAME         Мая
NAME         Александр
SURNAME      Вертинский
NAME         Николая
SURNAME      Вертинского
NAME         Евгенией
SURNAME      Скалацкой
SURNAME      Сколацкой
NAME         Надя
NAME         Саша
NAME         Евгения
PATRONYMIC   Степановна
NAME         Николай
PATRONYMIC   Петрович
NAME         Николая
SURNAME      Вертинского
NAME         Александра
SURNAME      Вертинского
SURNAME      Вер

In [55]:

suspects = {"тревоги", "монблана", "пьеро", "н"}

for doc_id in ["book_3539", "book_3677"]:
    doc = next(d for d in datasets["test"] if d["id"] == doc_id)
    text = doc["text"]

    for label, start, end in detect_person(text):
        value = text[start:end]

        if value.casefold() in suspects:
            context = text[max(0, start - 75):min(len(text), end + 75)]
            print(f"\n{doc_id} | {label}: {value!r}")
            print(context.replace("\n", " "))


book_3539 | NAME: 'Н'
казано Наде о брате. Спустя годы Александр случайно обнаружит упоминание о Н. Н. Вертинской в журнале «Театр и искусство», напишет ей, и выяснится, что

book_3539 | PATRONYMIC: 'Н'
ано Наде о брате. Спустя годы Александр случайно обнаружит упоминание о Н. Н. Вертинской в журнале «Театр и искусство», напишет ей, и выяснится, что эт

book_3539 | SURNAME: 'Монблана'
и, но даже самые яркие их имена блекнут рядом со сверкающей снежной шапкой Монблана в лице Фёдора Ивановича Шаляпина.  Живой бог русской музыки  В 1911–1912 г

book_3539 | SURNAME: 'Пьеро'
 Вертинского? Рассказывали, что генералиссимус любил слушать ариетки Брата Пьеро в часы отдыха — особенно песню «В синем и далеком океане». Легенда приписы


In [56]:

examples = ["Саша", "Bonnie", "George", "Walt", "H"]

for value in examples:
    print(f"{value:10} -> {replace_person(value, 'NAME', {})}")

cache = {}
assert replace_person("Саша", "NAME", cache) == replace_person("Саша", "NAME", cache)
assert len(replace_person("H", "NAME", {})) == 1

print("Проверки пройдены")


Саша       -> Валя
Bonnie     -> Nicole
George     -> John
Walt       -> Samuel
H          -> T
Проверки пройдены


In [57]:

problem_names = [
    "Робера", "Хайнца", "Фауста", "Паноса",
    "Грея", "Эдисона", "Хайди", "Ги",
    "Гаргантюа", "Сюнскэ", "Цзиньпина",
    "Штеффен", "Джейд", "Бодхидхарме"
]

for name in problem_names:
    print(f"\n{name}")
    for parse in morph.parse(name)[:3]:
        print(
            "  ",
            parse.normal_form,
            "|", parse.tag,
            "|", round(parse.score, 3)
        )



Робера
   робер | NOUN,anim,masc sing,gent | 0.222
   робер | NOUN,anim,masc sing,accs | 0.222
   робер | NOUN,inan,masc sing,gent | 0.222

Хайнца
   хайнец | NOUN,anim,masc sing,gent | 0.5
   хайнец | NOUN,anim,masc sing,accs | 0.5

Фауста
   фауст | NOUN,anim,masc,Sgtm,Surn sing,gent | 0.5
   фауст | NOUN,anim,masc,Sgtm,Surn sing,accs | 0.5

Паноса
   паноса | NOUN,anim,femn sing,nomn | 1.0

Грея
   греть | GRND,impf,tran pres | 0.333
   грей | NOUN,anim,masc,Sgtm,Surn sing,gent | 0.333
   грей | NOUN,anim,masc,Sgtm,Surn sing,accs | 0.333

Эдисона
   эдисон | NOUN,anim,masc,Sgtm,Surn sing,gent | 0.8
   эдисон | NOUN,anim,masc,Sgtm,Surn sing,accs | 0.2

Хайди
   хайти | VERB,perf,intr sing,impr,excl | 1.0

Ги
   ги | UNKN | 1.0

Гаргантюа
   гаргантю | NOUN,inan,masc sing,gent | 0.806
   гаргантюый | ADJS,Qual femn,sing | 0.156
   гаргантю | ADJF,Poss masc,sing,gent,Infr | 0.007

Сюнскэ
   сюнскэ | NOUN,inan,femn,Sgtm,Fixd sing,nomn | 0.167
   сюнскэ | NOUN,inan,femn,Sgtm,Fixd sing,g

In [58]:

examples = [
    "Робера", "Фауста", "Эдисона",
    "Грея", "Хайнца", "Лизка", "Мая"
]

for name in examples:
    try:
        replacement = replace_person("{}".format(name), "NAME", {})
        print(f"{name:12} -> {replacement}")
    except ValueError as e:
        print(f"{name:12} -> ОШИБКА: {e}")


Робера       -> Ладимира
Фауста       -> Гаврила
Эдисона      -> Ростислава
Грея         -> Арефия
Хайнца       -> Феликса
Лизка        -> Надежда
Мая          -> ОШИБКА: Неизвестная форма NAME: Мая


In [59]:
print(replace_person("Лизка", "NAME", {}))

print(replace_person(
    "Мая",
    "NAME",
    {},
    left_context="Смерть была тем миром, где кончались тщета мальчика "
))

Наина
Федосия


In [60]:

from collections import Counter

print("ВСЕ ОСТАВШИЕСЯ ОШИБКИ\n")

docs_by_id = {
    doc["id"]: doc["text"]
    for doc in datasets["test"]
}

for stage in ("GOLD", "PIPELINE"):
    print(f"\n{'=' * 15} {stage} {'=' * 15}")

    current = [
        (doc_id, message)
        for doc_id, error_stage, message in errors
        if error_stage == stage
    ]

    print("Количество:", len(current))

    for doc_id, message in current:
        text = docs_by_id[doc_id]
        value = message.rsplit(":", 1)[-1].strip()

        print(f"\n{doc_id}: {message}")

        if value:
            match = re.search(re.escape(value), text)

            if match:
                left = max(0, match.start() - 70)
                right = min(len(text), match.end() + 70)

                print("Контекст:",
                      text[left:right].replace("\n", " "))

print("\nВсего ошибок:", len(errors))


ВСЕ ОСТАВШИЕСЯ ОШИБКИ


=============== GOLD ===============
Количество: 14

book_3562: Неизвестная форма NAME: Ги
Контекст: дка на Ближнем Востоке — обсуждается в том числе и генсеком ООН Паном Ги Муном, главой Афганистана Хамидом Карзаем, премьер-министром Великобр

book_3615: Неизвестная форма NAME: Гаргантюа
Контекст: о имя: «Кто же этот розовощёкий индивид — обжора, пьянчуга и сластун? Гаргантюа, король дипсодов? Силач Фосс? Легендарный солдат Яшка Красная Рубашка

book_3714: Неизвестная форма NAME: Штеффен
Контекст: шение Россией международного права".  Представитель правительства ФРГ Штеффен Зайберт заявил, что президент США Барак Обама проведет двусторонную в

book_3758: Неизвестная форма NAME: Хе
Контекст: ении работы совместной промзоны Кэсон.  Президент Южной Кореи Пак Кын Хе также предложила КНДР открыть парк мира в демилитаризованной зоне на 

book_3776: Неизвестная форма NAME: Джейд
Контекст: коллег. Ранее на тему допинга в России высказывались прыгунья в длину Джейд Джон

In [61]:

# Проверки ЛР2 без повторного обучения модели

# 1. Формат входа и выхода
email = "ivan.petrov@gmail.com"

single = anon(email, ["EMAIL", 0, len(email)])
multiple = anon(email, [["EMAIL", 0, len(email)]])

assert isinstance(single, str)
assert single != email
assert multiple != email
assert single.endswith("@gmail.com")
assert multiple.endswith("@gmail.com")
assert anon("Обычный текст.", []) == "Обычный текст."

print("✓ Вход и выход anon()")

# 2. Случайность при повторных вызовах
for label, value in [
    ("EMAIL", email),
    ("INN", "770698959438"),
    ("PHONE", "+7 (999) 123-45-67"),
    ("CARD", "4561 2612 1234 5467")
]:
    results = {
        anon(value, [label, 0, len(value)])
        for _ in range(20)
    }
    assert len(results) > 1, label

print("✓ Случайность всех структурированных атрибутов")

# 3. Контрольные разряды и формат
for _ in range(500):
    inn = replace_inn("770698959438")
    card = replace_card("4561-2612-1234-5467")
    phone = replace_phone("+7 (999) 123-45-67")

    assert inn.startswith("7706") and valid_inn(inn)

    assert valid_card(card)
    assert digits_only(card)[:6] == "456126"
    assert digits_only(card)[-4:] == "5467"
    assert re.fullmatch(r"\d{4}-\d{4}-\d{4}-\d{4}", card)

    assert digits_only(phone)[:4] == "7999"
    assert re.fullmatch(r"\+7 \(999\) \d{3}-\d{2}-\d{2}", phone)

print("✓ 500 повторных проверок INN, CARD, PHONE")

# 4. Реакция на некорректные значения
bad_cases = [
    (replace_email, "bob..a@gmail.com"),
    (replace_email, ".foo@gmail.com"),
    (replace_email, "ivan@a..com"),
    (replace_inn, "770698959430"),
    (replace_card, "4561 2612 1234 5469"),
    (replace_phone, "abc123"),
]

for func, value in bad_cases:
    try:
        func(value)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Не отклонено: {value}")

# 5. Ошибки координат
try:
    anon("Иван", [["NAME", 0, 100]])
except ValueError:
    pass
else:
    raise AssertionError("Не отклонены неверные координаты")

print("✓ Некорректные значения и координаты")
print("\nПроверки структурированных атрибутов завершены.")


✓ Вход и выход anon()
✓ Случайность всех структурированных атрибутов
✓ 500 повторных проверок INN, CARD, PHONE
✓ Некорректные значения и координаты

Проверки структурированных атрибутов завершены.


In [64]:

import ast

print("=== СИНОНИМИЧЕСКОЕ ОБЕЗЛИЧИВАНИЕ ПДн ===")

text = input("Введите исходный текст: ")

print("\nВведите список ПДн в формате ЛР1.")
print('Пример: ["NAME", 0, 4]')
print('Несколько: [["NAME", 0, 4], ["SURNAME", 5, 11]]')
print("Нажмите Enter для автоматического распознавания.\n")

raw = input("Атрибуты ПДн: ").strip()

try:
    if raw:
        entities = ast.literal_eval(raw)
    else:
        entities = [
            e for e in detect_pii(text)
            if e[0] in {"NAME", "SURNAME", "EMAIL", "INN", "PHONE", "CARD"}
        ]

    print("\nНайденные ПДн:")
    items = [entities] if (
        isinstance(entities, (list, tuple))
        and len(entities) == 3
        and isinstance(entities[0], str)
    ) else entities

    for label, start, end in items:
        print(f"{label}: {text[start:end]!r} [{start}:{end}]")

    result = anon(text, entities)

    print("\nИСХОДНЫЙ ТЕКСТ:")
    print(text)

    print("\nОБЕЗЛИЧЕННЫЙ ТЕКСТ:")
    print(result)

except (ValueError, TypeError, NameError) as exc:
    print("\nОшибка:", exc)


=== СИНОНИМИЧЕСКОЕ ОБЕЗЛИЧИВАНИЕ ПДн ===


Введите исходный текст:  



Введите список ПДн в формате ЛР1.
Пример: ["NAME", 0, 4]
Несколько: [["NAME", 0, 4], ["SURNAME", 5, 11]]
Нажмите Enter для автоматического распознавания.



Атрибуты ПДн:  



Найденные ПДн:

ИСХОДНЫЙ ТЕКСТ:


ОБЕЗЛИЧЕННЫЙ ТЕКСТ:

